# GuitarSet F/E matched-seed augmentation repeats

Two new seeds, natural vs CQT transposition, both teachers: **8 new trainings**. Existing verified seed 20261119 is included only in the full aggregate. Read docs/p2/GUITARSET_TRANSPOSITION_REPEATS.md. Enable Internet and GPU; run in order.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, hashlib
project=Path('/kaggle/working/chord-practice-system')
if not project.exists():
    subprocess.run(['git','clone','https://github.com/PhatLavar/chord-practice-system.git',str(project)],check=True)
os.chdir(project)
subprocess.run(['git','pull','--ff-only','origin','main'],check=True)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True))
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-audio.txt'],check=True)
import torch
print('torch',torch.__version__,'CUDA',torch.cuda.is_available())
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests'],check=True)


In [ ]:
# Official pinned archives; about 696 MB total. No extraction to separate WAV files.
sys.path.insert(0,str(project/'src/acr'))
from audit_guitarset import download
from prepare_guitarset_cqt import load_cache
raw=Path('data/raw/guitarset')
download(raw)
config='configs/p2-guitarset-cqt-player.json'
cfg=json.loads(Path(config).read_text())
def prepare_cache(path, smoke=False):
    if not path.exists():
        command=[sys.executable,'src/acr/prepare_guitarset_cqt.py','--output',str(path),'--config',config]
        if smoke: command.append('--smoke')
        subprocess.run(command,check=True)
    # Checks archive identity recorded in cache, hashes, folds and train-only stats.
    songs,refs,meta=load_cache(path,cfg,smoke)
    print({'cache':str(path),'recordings':len(songs),'fold':cfg['fold'],'smoke':smoke})
    del songs, refs


In [ ]:
smoke_cache=Path('data/processed/guitarset-cqt-player-smoke')
prepare_cache(smoke_cache,smoke=True)
subprocess.run([sys.executable,'src/acr/repeat_guitarset.py','--cache',str(smoke_cache),'--output','runs/p2-guitarset-transpose-repeat-smoke','--device','cpu','--smoke'],check=True)


In [ ]:
cache=Path('data/processed/guitarset-cqt-player')
prepare_cache(cache)


In [ ]:
assert torch.cuda.is_available(), 'Select a GPU for the full repeats.'
run_root=Path('runs/p2-guitarset-transpose-repeat')
subprocess.run([sys.executable,'src/acr/repeat_guitarset.py','--cache',str(cache),'--output',str(run_root),'--device','cuda'],check=True)


In [ ]:
comparison=json.loads((run_root/'comparison.json').read_text())
print('Seeds',comparison['seeds'])
for teacher,metrics in comparison['aggregate'].items():
    result=metrics['tetrads']
    print(teacher,'natural mean/std',result['natural'],'transpose mean/std',result['transpose'],
          'paired delta (percentage points)',100*result['paired_delta']['mean'])
print('Inspect all metrics and per-quality results, not only the primary mean. Development validation, one held-out player; E uses gold boundaries.')


In [ ]:
import shutil
archive=shutil.make_archive('/kaggle/working/p2-guitarset-transpose-repeat-results','zip',project,'runs/p2-guitarset-transpose-repeat')
print(archive)


Save the executed notebook and download **p2-guitarset-transpose-repeat-results.zip** into local runs/imports/. The archive includes all eight new checkpoints, histories, predictions, configs and provenance plus the three-seed comparison. Historical seed-20261119 metric file hashes are in plan.json; its existing checkpoints are not duplicated. Completed jobs require matching provenance; incomplete jobs are preserved and rejected. Smoke output is not included in performance aggregation.
